In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("All India Consumer Price Index.csv")
print(df.head())

In [ ]:
df.info()

In [ ]:
print(df.isnull())

In [ ]:
print(df.isnull().sum())

In [ ]:
cpi_columns = [
"Cereals and products","Meat and fish","Egg","Milk and products",
"Oils and fats","Fruits","Vegetables","Pulses and products",
"Sugar and Confectionery","Spices","Non-alcoholic beverages",
"Prepared meals, snacks, sweets etc.","Food and beverages",
"Pan, tobacco and intoxicants","Clothing","Footwear",
"Clothing and footwear","Housing","Fuel and light",
"Household goods and services","Health","Transport and communication",
"Recreation and amusement","Education","Personal care and effects",
"Miscellaneous","General index"
]
for col in cpi_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns
for col in numeric_columns:
    df[col] = df[col].fillna(df[col].mean())

categorical_columns = df.select_dtypes(include="object").columns
for col in categorical_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

In [ ]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())

In [ ]:
outlier_counts = {}
for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_counts[col] = len(outliers)
print(outlier_counts)

In [ ]:
visualization_columns = [
"Food and beverages","Fuel and light","Health",
"Transport and communication","Education","General index"
]
for col in visualization_columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[col])
    plt.title("Outlier Detection - " + col)
    plt.xlabel(col)
    plt.show()

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(",", "")
)
print(df.columns)

In [ ]:
scaler = MinMaxScaler()
numeric_columns = df.select_dtypes(include=np.number).columns
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
print(df.head())

In [ ]:
df.to_excel("cleaned_consumer_price_index.xlsx", index=False)